# IndicF5: Malayalam evaluation for Mixlumo (offline voice option)

**Before running (one time):**
1. Sign in at **huggingface.co**, open **https://huggingface.co/ai4bharat/IndicF5** and accept the access conditions.
   **Take a screenshot of the conditions text** and send it to the Mixlumo developer (needed for the licence check).
2. Create a token: huggingface.co → Settings → **Access Tokens** → New token → type **Read**. Copy it.
3. Kaggle: **Add-ons → Secrets → Add a new secret**, label **`HF_TOKEN`**, paste the token, tick it for this notebook.
   (Colab: the key icon on the left → `HF_TOKEN`, enable notebook access.) The token is never printed or saved.
4. Session options: **Accelerator GPU T4 x2**, **Internet On**. Then **Run All**. At the end download **`indicf5_eval.zip`**.

Use a **fresh session**: if this Kaggle session ran another notebook before (e.g. the Maya test), first click
**Run → Factory reset** so leftover packages from that run can't interfere.

What it measures with the real model (nothing simulated): loads at all on current Kaggle; Malayalam speech cloned from
two **synthetic** reference voices (made by Maya 2 Native, so no real person's voice); colloquial Malayalam; numbers;
speaker similarity; GPU memory; speed with 32 and 16 steps; half precision; **whether it fits a 4 GB GPU** (like a
GTX 1650); and CPU-only speed.

In [ ]:
# 1. Environment
import os, subprocess, sys, json, time, platform, shutil
OUT = "/kaggle/working/indicf5_eval" if os.path.isdir("/kaggle/working") else os.path.abspath("indicf5_eval")
os.makedirs(f"{OUT}/clips", exist_ok=True)
RESULTS = {"started": time.strftime("%Y-%m-%d %H:%M:%S"), "python": platform.python_version(), "steps": {}}
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() if shutil.which("nvidia-smi") else ""
RESULTS["gpu"] = smi
print("Python", platform.python_version(), "| disk free GB:", shutil.disk_usage("/").free // 2**30)
if not smi:
    raise SystemExit("NO GPU ATTACHED. Kaggle: Session options > Accelerator > GPU T4 x2. Colab: Runtime > Change runtime type > T4 GPU.")
print("GPU:", smi)

In [ ]:
# 2. Install IndicF5 (pinned commit) without its old pins (numpy<=1.26.4 has no Python 3.13 build); keep the platform's PyTorch
if "transformers" in sys.modules:
    raise SystemExit("transformers is already loaded: Kaggle Run > Restart & clear cell outputs (Colab: Runtime > Restart session), then Run all.")
INDICF5_COMMIT = "13f7c4d"
t = time.time()
# Lock the platform's PyTorch: pip must never up/downgrade it (that broke torchvision in an earlier run)
from importlib import metadata
LOCK = {}
for pkg in ("torch", "torchaudio"):
    try:
        LOCK[pkg] = metadata.version(pkg)
    except metadata.PackageNotFoundError:
        pass
def _mm(v):
    return ".".join(v.split("+")[0].split(".")[:2])
leftovers = [p for p in ("maya3",) if p in {d.metadata["Name"].lower() for d in metadata.distributions()}]
if leftovers or ("torch" in LOCK and "torchaudio" in LOCK and _mm(LOCK["torch"]) != _mm(LOCK["torchaudio"])):
    raise SystemExit(f"This session has leftovers from an earlier run (torch {LOCK.get('torch')}, torchaudio "
                     f"{LOCK.get('torchaudio')}, other packages: {leftovers}). Kaggle: Run > Factory reset "
                     "(Colab: Runtime > Disconnect and delete runtime), then Run all.")
open(f"{OUT}/constraints.txt", "w").write("\n".join(f"{k}=={v}" for k, v in LOCK.items()) + "\n")
print("keeping", LOCK)
cmds = [
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", f"git+https://github.com/AI4Bharat/IndicF5@{INDICF5_COMMIT}"],
    [sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0", "ema_pytorch>=0.5.2", "torchdiffeq",
     "x_transformers>=1.31.14", "vocos", "pydub", "jieba", "pypinyin", "cached_path", "speechbrain", "wandb", "datasets",
     "accelerate>=0.33.0", "librosa", "matplotlib", "soundfile", "safetensors", "hydra-core", "tomli",
     "transformers_stream_generator", "-c", f"{OUT}/constraints.txt"],
]
log = ""
for c in cmds:
    r = subprocess.run(c, capture_output=True, text=True)
    log += r.stdout[-1500:] + r.stderr[-3000:]
    if r.returncode != 0:
        print(log); raise SystemExit("install failed: see above")
print(log[-2500:])
# transformers imports torchvision when present; one that doesn't match PyTorch breaks it ("torchvision::nms does not
# exist"). IndicF5 doesn't use torchvision, so remove a mismatched one.
tv = subprocess.run([sys.executable, "-c", "import torchvision"], capture_output=True, text=True)
if tv.returncode != 0 and "No module named" not in tv.stderr:
    print("torchvision doesn't match this PyTorch: removing it (not needed by IndicF5)")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision"], capture_output=True, text=True)
chk = subprocess.run([sys.executable, "-c", "import torch, torchaudio, transformers, vocos; import f5_tts.infer.utils_infer; "
                      "print('imports OK: torch', torch.__version__, '| torchaudio', torchaudio.__version__, "
                      "'| transformers', transformers.__version__)"], capture_output=True, text=True)
print(chk.stdout, chk.stderr[-3000:])
RESULTS["steps"]["install"] = {"ok": chk.returncode == 0, "seconds": round(time.time() - t, 1),
                               "imports": (chk.stdout + chk.stderr)[-1500:]}
assert chk.returncode == 0, "imports fail: see the error above"

In [ ]:
# 3. Hugging Face access (gated model). The token comes from Kaggle/Colab secrets and is never printed.
def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        pass
    return os.environ.get("HF_TOKEN")

tok = hf_token()
if not tok:
    raise SystemExit("No HF_TOKEN secret found. Add it (Kaggle: Add-ons > Secrets; Colab: key icon) and enable it for this notebook.")
os.environ["HF_TOKEN"] = tok
from huggingface_hub import HfApi, snapshot_download
try:
    info = HfApi().model_info("ai4bharat/IndicF5", token=tok)
    RESULTS["model_info"] = {"sha": info.sha, "gated": getattr(info, "gated", None),
                             "license": (info.card_data or {}).get("license") if info.card_data else None,
                             "files": sorted(s.rfilename for s in info.siblings)}
    print("access OK | revision", info.sha, "| licence tag", RESULTS["model_info"]["license"])
    print("files:", RESULTS["model_info"]["files"])
except Exception as e:
    raise SystemExit(f"Cannot access ai4bharat/IndicF5 ({type(e).__name__}). Open the model page while signed in, accept "
                     "the conditions, check the token is a Read token, then Run all again.")

In [ ]:
# 4. Load on GPU; keep a copy of the model's own code (for the licence/code review)
import torch, numpy as np, soundfile as sf, inspect
from transformers import AutoModel

snap = snapshot_download("ai4bharat/IndicF5", token=tok, allow_patterns=["*.py", "*.json", "*.md", "LICENSE*", "*.txt"])
os.makedirs(f"{OUT}/model_code", exist_ok=True)
for f in os.listdir(snap):
    if f.endswith((".py", ".json", ".md", ".txt")) or f.startswith("LICENSE"):
        shutil.copy(os.path.join(snap, f), f"{OUT}/model_code/{f}")

torch.cuda.reset_peak_memory_stats(); t = time.time()
model = AutoModel.from_pretrained("ai4bharat/IndicF5", trust_remote_code=True, token=tok)
try:
    model = model.to("cuda")
except Exception as e:
    print("(.to('cuda') not applied:", e, ")")
load_s = round(time.time() - t, 1)
params = list(model.parameters())
RESULTS["load"] = {"seconds": load_s, "class": type(model).__name__,
                   "params_millions": round(sum(p.numel() for p in params) / 1e6, 1),
                   "dtypes": sorted({str(p.dtype) for p in params}), "devices": sorted({str(p.device) for p in params}),
                   "gpu_gb_after_load": round(torch.cuda.memory_allocated() / 2**30, 2),
                   "call_signature": str(inspect.signature(model.forward)) if hasattr(model, "forward") else None}
print(json.dumps(RESULTS["load"], indent=1))
json.dump(RESULTS, open(f"{OUT}/results.json", "w"), indent=1, default=str)

### 5. Reference voices (synthetic) and test sentences

Two reference clips made by **Maya 2 Native** voices *Ahana* (female) and *Anirban* (male), with their exact transcript.
They are synthetic, so no real person's voice is cloned. Test sentences were written for this evaluation.

In [ ]:
import base64, io
REF_TEXT = ("മഴ പെയ്യുമ്പോൾ ഞാൻ ജനലിനരികിൽ ഇരുന്ന് ചായ കുടിക്കും. പുറത്ത് മരങ്ങൾ കാറ്റിൽ ആടുന്നു. "
            "ആ ശബ്ദം കേൾക്കാൻ എനിക്ക് വളരെ ഇഷ്ടമാണ്.")
REFS = {"female": "...", "male": "..."}
REF_PATHS = {}
for k, b in REFS.items():
    w, sr = sf.read(io.BytesIO(base64.b64decode(b)), dtype="float32")
    REF_PATHS[k] = f"{OUT}/clips/ref_{k}.wav"; sf.write(REF_PATHS[k], w, sr)
    print(k, "reference:", round(len(w) / sr, 2), "s at", sr, "Hz")

S = {
    "greeting":   "നമസ്കാരം, മിക്സ്ലൂമോ വോയ്സ് സ്റ്റുഡിയോയിലേക്ക് സ്വാഗതം.",                 # Hello, welcome to Mixlumo Voice Studio.
    "weather":    "ഇന്ന് നല്ല കാലാവസ്ഥയാണ്, നമുക്ക് പുറത്ത് നടക്കാൻ പോകാം.",             # The weather is nice today, let's go for a walk.
    "kerala":     "കേരളം ദൈവത്തിന്റെ സ്വന്തം നാടാണ്.",                                     # Kerala is God's own country.
    "app":        ("ഈ ആപ്ലിക്കേഷൻ ഉപയോഗിച്ച് നിങ്ങൾക്ക് പാട്ടുകളിൽ നിന്ന് ശബ്ദം വേർതിരിക്കാനും "
                   "മലയാളത്തിൽ സംസാരിക്കുന്ന ശബ്ദം സൃഷ്ടിക്കാനും കഴിയും."),              # With this app you can separate vocals and create Malayalam speech.
    "colloquial": "എടാ, നീ എവിടെയാ? ഞാൻ ഇവിടെ കാത്തിരിക്കുവാ, പെട്ടെന്ന് വാ.",           # Hey, where are you? I'm waiting here, come quickly. (spoken style)
    "numbers":    "ഈ ടിക്കറ്റിന് 250 രൂപയാണ്, ബസ് രാവിലെ 7:30-ന് പുറപ്പെടും.",              # digits: no text normaliser in IndicF5
    "short":      "ശരി.",                                                                    # OK.
}

In [ ]:
# 6. Generate: every sentence with both reference voices
def stats(w):
    w = np.asarray(w, np.float32)
    if len(w) == 0:
        return {"empty": True}
    n = len(w) // 2400
    rms = np.sqrt((w[: n * 2400].reshape(n, 2400) ** 2).mean(1)) if n else np.array([0.0])
    return {"audio_s": round(len(w) / 24000, 2), "peak": round(float(np.abs(w).max()), 3),
            "rms_dbfs": round(float(20 * np.log10(np.sqrt(np.mean(w ** 2)) + 1e-9)), 1),
            "clipped_samples": int((np.abs(w) >= 0.999).sum()), "finite": bool(np.isfinite(w).all()),
            "silent_fraction": round(float((rms < 0.003).mean()), 2)}

def synth(text, ref, **kw):
    out = model(text, ref_audio_path=REF_PATHS[ref], ref_text=REF_TEXT, **kw)
    if isinstance(out, (tuple, list)):
        out = out[0]
    out = np.asarray(out.detach().cpu() if hasattr(out, "detach") else out)
    if out.dtype == np.int16:
        out = out.astype(np.float32) / 32768.0
    return out.astype(np.float32).reshape(-1)

CLIPS = []
def clip(fname, text, ref, **kw):
    rec = {"file": fname, "text": text, "ref": ref, **kw}
    try:
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); t = time.time()
        w = synth(text, ref, **kw)
        torch.cuda.synchronize(); dt = time.time() - t
        sf.write(f"{OUT}/clips/{fname}.wav", w, 24000)
        rec.update(stats(w)); rec["gen_seconds"] = round(dt, 2)
        rec["rtf"] = round(dt / max(len(w) / 24000, 1e-6), 3)
        rec["peak_gpu_gb"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {e}"[:800]
    CLIPS.append(rec)
    print(fname, {k: rec.get(k) for k in ("audio_s", "gen_seconds", "rtf", "peak_gpu_gb", "error")})
    return rec

def save():
    RESULTS["clips"] = CLIPS
    json.dump(RESULTS, open(f"{OUT}/results.json", "w"), indent=1, default=str)

clip("00_warmup", S["kerala"], "female")          # first call includes one-time setup
for ref in ("female", "male"):
    for k, text in S.items():
        clip(f"10_{k}_{ref}", text, ref)
save()

In [ ]:
# 7. Speed vs quality: 16 diffusion steps instead of 32 (if the model's call accepts nfe_step)
sig = RESULTS["load"].get("call_signature") or ""
if "nfe_step" in sig or "kwargs" in sig:
    for k in ("app", "colloquial"):
        clip(f"20_{k}_female_nfe16", S[k], "female", nfe_step=16)
else:
    print("nfe_step not accepted by this model's call; skipped")
save()

In [ ]:
# 8. Speaker similarity (SpeechBrain ECAPA, the same speaker encoder Maya uses): each clone vs its own reference
#    and vs the other reference. A working clone is closer to its own reference.
from speechbrain.inference.speaker import EncoderClassifier
import torchaudio
enc = EncoderClassifier.from_hparams("speechbrain/spkrec-ecapa-voxceleb", run_opts={"device": "cuda"},
                                     savedir="/tmp/ecapa")
def emb(path):
    w, sr = torchaudio.load(path)
    w = torchaudio.functional.resample(w.mean(0, keepdim=True), sr, 16000)
    e = enc.encode_batch(w.cuda()).reshape(-1)
    return torch.nn.functional.normalize(e, dim=0).cpu().numpy()
E_REF = {k: emb(p) for k, p in REF_PATHS.items()}
SIM = []
for c in CLIPS:
    if "error" in c or c["file"] == "00_warmup" or c.get("audio_s", 0) < 1.0:
        continue
    e = emb(f"{OUT}/clips/{c['file']}.wav")
    own, other = c["ref"], "male" if c["ref"] == "female" else "female"
    SIM.append({"file": c["file"], "to_own_ref": round(float(e @ E_REF[own]), 3),
                "to_other_ref": round(float(e @ E_REF[other]), 3)})
    print(SIM[-1])
RESULTS["similarity"] = SIM
RESULTS["similarity_refs_to_each_other"] = round(float(E_REF["female"] @ E_REF["male"]), 3)
save()

In [ ]:
# 9. Half precision (fp16): less memory and faster on GPUs like the GTX 1650, if quality holds
fp16 = {}
try:
    model.half()
    for k in ("app", "colloquial"):
        r = clip(f"30_{k}_female_fp16", S[k], "female")
        fp16[k] = {x: r.get(x) for x in ("rtf", "peak_gpu_gb", "finite", "error")}
except Exception as e:
    fp16["error"] = f"{type(e).__name__}: {e}"[:800]
RESULTS["fp16"] = fp16
print(fp16)
try:
    model.float()
except Exception:
    pass
save()

In [ ]:
# 10. Does it fit a 4 GB GPU (like a GTX 1650)? A separate process is limited to 3.3 GB of this GPU's memory
#     (a 4 GB card has ~3.3-3.5 GB free after the driver/CUDA context), loads the model, and speaks a long sentence.
LIMIT_SCRIPT = "\n".join([
    "import json, sys, time, os, numpy as np, torch, soundfile as sf",
    "from transformers import AutoModel",
    "out, ref, ref_text, text, half = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4], sys.argv[5] == '1'",
    "total = torch.cuda.get_device_properties(0).total_memory",
    "torch.cuda.set_per_process_memory_fraction(3.3 * 2**30 / total)",
    "rec = {'limit_gb': 3.3, 'half': half}",
    "try:",
    "    m = AutoModel.from_pretrained('ai4bharat/IndicF5', trust_remote_code=True, token=os.environ.get('HF_TOKEN'))",
    "    m = m.to('cuda')",
    "    if half: m = m.half()",
    "    t = time.time(); w = m(text, ref_audio_path=ref, ref_text=ref_text); torch.cuda.synchronize(); dt = time.time() - t",
    "    w = np.asarray(w); w = w.astype(np.float32) / 32768.0 if w.dtype == np.int16 else w.astype(np.float32)",
    "    rec.update(ok=bool(np.isfinite(w).all() and len(w) > 24000), audio_s=round(len(w) / 24000, 2), gen_seconds=round(dt, 2),",
    "               peak_gb=round(torch.cuda.max_memory_allocated() / 2**30, 2))",
    "    sf.write(out + '/clips/40_fits_4gb_' + ('fp16' if half else 'fp32') + '.wav', w, 24000)",
    "except BaseException as e:",
    "    rec.update(ok=False, error=(type(e).__name__ + ': ' + str(e))[:800])",
    "print('RESULT ' + json.dumps(rec))",
])
open(f"{OUT}/limit_test.py", "w").write(LIMIT_SCRIPT)
RESULTS["fits_4gb"] = []
for half in ("0", "1"):
    p = subprocess.run([sys.executable, f"{OUT}/limit_test.py", OUT, REF_PATHS["female"], REF_TEXT, S["app"], half],
                       capture_output=True, text=True, timeout=1800, env=dict(os.environ))
    line = next((l for l in p.stdout.splitlines() if l.startswith("RESULT ")), None)
    rec = json.loads(line[7:]) if line else {"half": half == "1", "ok": False, "error": "process died", "stderr": p.stderr[-1500:]}
    RESULTS["fits_4gb"].append(rec); print(rec)
save()

In [ ]:
# 11. CPU only (no GPU at all): speed of one short and one medium sentence. Kaggle's CPU is slower than most desktops.
CPU_SCRIPT = "\n".join([
    "import json, sys, time, os, numpy as np, torch",
    "from transformers import AutoModel",
    "ref, ref_text = sys.argv[1], sys.argv[2]; texts = json.loads(sys.argv[3])",
    "rec = {'threads': torch.get_num_threads(), 'runs': []}",
    "try:",
    "    t = time.time(); m = AutoModel.from_pretrained('ai4bharat/IndicF5', trust_remote_code=True, token=os.environ.get('HF_TOKEN'))",
    "    rec['load_seconds'] = round(time.time() - t, 1)",
    "    for text in texts:",
    "        t = time.time(); w = np.asarray(m(text, ref_audio_path=ref, ref_text=ref_text)); dt = time.time() - t",
    "        rec['runs'].append({'text_chars': len(text), 'audio_s': round(len(w) / 24000, 2), 'gen_seconds': round(dt, 1),",
    "                            'rtf': round(dt / max(len(w) / 24000, 1e-6), 2)})",
    "except BaseException as e:",
    "    rec['error'] = (type(e).__name__ + ': ' + str(e))[:800]",
    "print('RESULT ' + json.dumps(rec))",
])
open(f"{OUT}/cpu_test.py", "w").write(CPU_SCRIPT)
env = dict(os.environ, CUDA_VISIBLE_DEVICES="")
try:
    p = subprocess.run([sys.executable, f"{OUT}/cpu_test.py", REF_PATHS["female"], REF_TEXT,
                        json.dumps([S["kerala"], S["weather"]])], capture_output=True, text=True, timeout=2400, env=env)
    line = next((l for l in p.stdout.splitlines() if l.startswith("RESULT ")), None)
    RESULTS["cpu"] = json.loads(line[7:]) if line else {"error": "process died", "stderr": p.stderr[-1500:]}
except subprocess.TimeoutExpired:
    RESULTS["cpu"] = {"error": "timeout after 40 min"}
print(RESULTS["cpu"]); save()

In [ ]:
# 12. Summary + zip for download
ok = [c for c in CLIPS if "error" not in c and c["file"] != "00_warmup"]
RESULTS["summary"] = {
    "clips_ok": len(ok), "clips_failed": sum(1 for c in CLIPS if "error" in c),
    "median_rtf_gpu_fp32": round(float(np.median([c["rtf"] for c in ok if c["file"].startswith("10_")])), 3) if ok else None,
    "max_peak_gpu_gb": max((c.get("peak_gpu_gb", 0) for c in ok), default=None),
    "fits_4gb": [{k: r.get(k) for k in ("half", "ok", "peak_gb", "gen_seconds", "error")} for r in RESULTS.get("fits_4gb", [])],
    "cpu": RESULTS.get("cpu"),
}
RESULTS["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
save()
print(json.dumps(RESULTS["summary"], indent=1, ensure_ascii=False))
shutil.make_archive(OUT, "zip", OUT)
print("Download:", OUT + ".zip", round(os.path.getsize(OUT + ".zip") / 2**20, 1), "MB")